# Notebook ready to use for general NLP question

- loads a model (classification one - encoder)
- loads data (from huggingface)
- tokenize it in order to feed it into the model
- finetune train on some smaller data that we have / get
- test it
- save it in `.csv` format

Porpuse : learning and some start point for the **WAICC Hackathon**

NEXT CELL: config to understand how to exec the pipeline

## Hackathon Pipeline

### 0. Before Running Anything
1. **Enable GPU:** `Runtime` -> `Change runtime type` -> **T4 GPU**.
2. **Update Constants (Cells 5 & 6):**
   - `TASK`: Name of the competition task (used for saving folders & `submission.csv`).
   - `MODEL_NAME`: Use `"dumitrescustefan/bert-base-romanian-cased-v1"` or `"xlm-roberta-base"` for Romanian, or `"roberta-base"` / `"microsoft/deberta-v3-base"` for English.
   - `NUM_LABELS` & `id2label` / `label2id`: Match the exact number of classes in the dataset.
   - `MAX_SEQ_LEN`: `128` for short tweets/headlines, `256` for paragraphs, `512` for full articles.

---

### Mode A: Full Training Run (Start of Hackathon / New Model)
Run cells in order from top to bottom, **skipping** the *"Load from Drive"* cell:
1. **Setup & Drive:** Run `!pip install`, `!nvidia-smi`, `drive.mount`, and Imports.
2. **Config & Data:** Run Task Constants, Hyperparameters, and Data Loading (replace `load_dataset` with `pd.read_csv` + `train_test_split` when using competition CSV files).
3. **Tokenization & Setup:** Run `AutoTokenizer`, `preprocess_text` (`.map`), `DataCollatorWithPadding`, `compute_metrics`, `AutoModelForSequenceClassification`, `TrainingArguments`, and `Trainer`.
4. **Train & Save:** Run `trainer.train()` followed immediately by the **Save to Drive** cell (`trainer.save_model` + `tokenizer.save_pretrained`).
5. **Test & Submit:** Run the **Inference (`pipeline`)** cell and the **Submission CSV** cell.

---

### Mode B: Colab Disconnected After Training (Resume Without Retraining)
If your runtime resets *after* the model was already saved to Google Drive, **do NOT run `trainer.train()` again**:
1. Run **Imports** + **Task Constants & Hyperparameters**.
2. Run **Data Loading** (only needed if generating `submission.csv` on the test set).
3. Run the **Load from Drive** cell (`drive.mount` + `AutoTokenizer.from_pretrained` + `AutoModelForSequenceClassification.from_pretrained`).
4. Jump directly to the **Inference (`pipeline`)** and **Submission CSV** cells.
   *(Note: Only re-run `DataCollator`, `compute_metrics`, and `Trainer(...)` if you specifically want to call `trainer.evaluate()`).*

In [1]:
!pip install -q transformers datasets evaluate accelerate numpy

In [2]:
!nvidia-smi

Sun Nov 15 11:16:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate

import numpy as np


In [7]:
# define the task through 5 constants

# the task will be fake news detection - so the labels that we will have -> 1/0

TASK = "fake_news"
MODEL_NAME = "roberta-base"
DATASET_NAME = "GonzaloA/fake_news"
NUM_LABELS = 2
METRIC_NAME = "f1"

# f1-score -> metric used to penalize lazy models
# in order to get balanced accury over the whole dataset
# in balanced data scenarios -> should be the same as the accuracy

In [9]:
# training hyperparams
MAX_SEQ_LEN = 512 # might want to use smaller on task that involve only sentence
BATCH_SIZE = 16
NUM_EPOCHS = 3
LEARNING_RATE = 2e-5
# OUTPUT_DIR = f"test_trainer_{TASK}"

OUTPUT_DIR = f"/content/drive/MyDrive/hackathon_nlp_{TASK}"
# we use this in order to save the finetuned models on google drive -> no disconnect from local servers of colab



In [10]:
raw_dataset = load_dataset(DATASET_NAME)

raw_dataset["train"] = raw_dataset["train"].shuffle(seed=42).select(range(2000))
raw_dataset["validation"] = raw_dataset["validation"].shuffle(seed=42).select(range(1000))
raw_dataset["test"] = raw_dataset["test"].shuffle(seed=42).select(range(1000))


print(raw_dataset["train"][0])

README.md:   0%|          | 0.00/6.73k [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


dataset_infos.json:   0%|          | 0.00/1.03k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 38.8MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 13.0MB            

... (3 outputs left out)

Generating train split:   0%|          | 0/24353 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/8117 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/8117 [00:00<?, ? examples/s]

{'Unnamed: 0': 4694, 'title': 'Flap in Congress over gender identity sinks spending bill', 'text': 'WASHINGTON (Reuters) - The rancorous political debate over sexual identity unexpectedly prompted the Republican-controlled House of Representatives to reject an energy and water spending bill on Thursday after Democrats attached an amendment to protect the rights of transgender people. The legislation, which would provide funds for the U.S. Army Corps of Engineers and the Department of Energy in the fiscal year that starts Oct. 1, failed on a 112-305 vote, with 130 Republicans and 175 Democrats opposing the legislation. The outcome was such a surprise that the House Appropriations Committee initially announced the bill had passed and was forced to retract the statement.  Democrats blamed Republicans for opposing a Democratic amendment to bar federal contractors from government work if they discriminate against the lesbian, gay, bisexual and transgender (LGBT) community. Democrats failed 

In [11]:
id2label = {
    0: "REAL",
    1: "FAKE"
}

label2id = {
    "REAL": 0,
    "FAKE": 1
}

# tokenize the text

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# did this in order to get the correct tokenizer for the model we are using

example_text = "tung tung sahur is facing prison over Jensen affair"
tokens = tokenizer.tokenize(example_text)
print(tokens)



config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

['t', 'ung', 'Ġt', 'ung', 'Ġsa', 'hur', 'Ġis', 'Ġfacing', 'Ġprison', 'Ġover', 'ĠJensen', 'Ġaffair']


In [12]:


def preprocess_text(examples):
  """
  tokenize a batch of text examples
  args:
    examples: a batch of dataset rows (dict of lists)
  """

  return tokenizer(
      examples["text"],
      truncation=True,
      padding=False,
      max_length=MAX_SEQ_LEN
  )

tokenized_dataset = raw_dataset.map(
    preprocess_text,
    batched=True
)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [13]:
print(tokenized_dataset["train"].column_names)

['Unnamed: 0', 'title', 'text', 'label', 'input_ids', 'attention_mask']


In [14]:
# we will handle padding manually through data collator
# pads the whole batch to the longest seq in that batch

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)



In [15]:
# define the evaluation metrics -> in our case the f1-score and accuracy

f1_metric = evaluate.load(METRIC_NAME)
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
  """
    called by trainer at the end of each evaluation step

    args:
      eval_pred: a tuple (precictions, labels)

    returns:
      dict with f1 and accuracy metrics
  """


  logits, labels = eval_pred
  predictions = np.argmax(logits, axis = 1)

  f1 = f1_metric.compute(
      predictions=predictions,
      references=labels,
      average="weighted" # accounts imbalance in the dataset
  )

  accuracy = accuracy_metric.compute(
      predictions=predictions,
      references=labels
  )

  return {
      "f1": f1["f1"],
      "accuracy": accuracy["accuracy"]
  }



In [16]:
# load the pretrained model finally

# AutoModelForSequenceClassification adds the classification head (linear layer) on top the encode

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [17]:
# define training config


# TrainingArguments is the config object for the Trainer
# these are the hyperparams we tune

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # base hyperparams
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    fp16 = True,
    weight_decay = 0.01,

    # evaluation and saving
    eval_strategy = "epoch",
    save_strategy = "epoch",
    save_total_limit = 1, # only save one checkpoint
    load_best_model_at_end = True, # gives us the best checkpoint of the model
    metric_for_best_model = METRIC_NAME,

    # logging
    logging_steps = 50, # log every 50 steps
    report_to = "none"
)

In [18]:
# create the trainer model

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [19]:
# train the model (more like finetune)

train_result = trainer.train()

print(f"training loss: {train_result.training_loss:.4f}")


Epoch,Training Loss,Validation Loss,F1,Accuracy
1,0.083582,0.043478,0.974040,0.974000
2,0.065344,0.096338,0.975040,0.975000
3,0.023429,0.051304,0.978011,0.978000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

training loss: 0.0770


In [20]:
FINAL_MODEL_DIR = f"/content/drive/MyDrive/final_model_{TASK}"

trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

print("MODEL and TOKENIZER saved")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

MODEL and TOKENIZER saved


The next cell is used to load a saved checkpoint / final state model that was already finetuned.
We get it from drive.

In [ ]:
from google.colab import drive
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

TASK = "fake_news"
drive.mount("/content/drive")
FINAL_MODEL_DIR = f"/content/drive/MyDrive/final_model_{TASK}"

tokenizer = AutoTokenizer.from_pretrained(FINAL_MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(FINAL_MODEL_DIR)

In [21]:
# evaluate the test dataset

test_results = trainer.evaluate(tokenized_dataset["test"])

for key, value in test_results.items():
  if isinstance(value, float):
    print(f"  {key} : {value:.4f}")

Training Loss,Validation Loss,Epoch,F1,Accuracy
0.023429,0.021620,3,0.989001,0.989000


  eval_loss : 0.0216
  eval_f1 : 0.9890
  eval_accuracy : 0.9890


In [23]:
# run inference through pipeline
from transformers import pipeline
import torch

classifier = pipeline(
    "text-classification",
    model = model,
    tokenizer = tokenizer,
    device = 0 if torch.cuda.is_available() else -1,
    truncation = True,
    max_length = MAX_SEQ_LEN,
)

test_samples = [
    # real
    "The European Central Bank kept benchmark interest rates unchanged on Thursday as inflation hovered near its 2% target across the eurozone.",
    "Researchers at the University of Tokyo demonstrated a new biodegradable polymer capable of dissolving in seawater within four weeks.",
    "Local transit authorities announced schedule adjustments for three commuter rail lines due to weekend track maintenance.",
    "Quarterly earnings for major semiconductor manufacturers rose 14% year-over-year driven by sustained data center demand.",

    # fake
    "SHOCKING: Doctors hate this one simple kitchen spice that completely cures diabetes overnight without medication!",
    "BREAKING: Secret government documents prove the moon is actually a hollow metallic station built 10,000 years ago!",
    "Share before they delete this! Drinking boiled garlic water every morning makes you 100% immune to all viruses!",

    # subtle fake
    "GENEVA (Reuters) - The World Health Organization officially recommended on Tuesday that adults replace two hours of daily sleep with moderate caffeine intake to improve cardiovascular longevity.",
    "WASHINGTON (AP) - NASA confirmed that Earth's magnetic poles will permanently reverse next Tuesday, causing all compasses and GPS satellites to point south until 2030.",
    "BRUSSELS - Under a new directive passed unanimously on Friday, European Union member states will phase out all physical banknotes in favor of biological microchip implants by January."
]

results = classifier(test_samples)

for res in results:
  print(f"prediction: {res['label']}, confidence: {res['score']:.4f}")



prediction: FAKE, confidence: 0.9995
prediction: REAL, confidence: 0.9169
prediction: FAKE, confidence: 0.8820
prediction: FAKE, confidence: 0.8356
prediction: REAL, confidence: 0.9991
prediction: REAL, confidence: 0.9988
prediction: REAL, confidence: 0.9994
prediction: FAKE, confidence: 0.9998
prediction: FAKE, confidence: 0.9997
prediction: FAKE, confidence: 0.9998


In [ ]:
# generate the final submission csv using the trainer
# this or the next cell should do the same thing
import pandas as pd

raw_predicts = trainer.predict(tokenized_dataset["test"])

predicted_classes = np.argmax(raw_predicts.predictions, axis = 1)

submission = pd.DataFrame({
    "id": range(len(predicted_classes)),
    "label": predicted_classes
})

submission.to_csv("submission.csv", index = False)

print(submission.head())


In [24]:
# submission csv made through pipeline (a lot more easy to use)
import pandas as pd
from transformers import pipeline
import torch

classifier = pipeline(
    "text-classification",
    model = model,
    tokenizer = tokenizer,
    device = 0 if torch.cuda.is_available() else -1,
    truncation = True,
    max_length = MAX_SEQ_LEN
)

test_texts = list(raw_dataset["test"]["text"])
preds = classifier(test_texts, batch_size = BATCH_SIZE)

predicted_classes = [model.config.label2id[pred["label"]] for pred in preds]

submission = pd.DataFrame({
    "id": range(len(predicted_classes)),
    "label": predicted_classes
})

# this is a local save
# submission.to_csv("submission.csv", index = False)

# this is saving inside the drive
submission.to_csv(f"/content/drive/MyDrive/hackathon_nitronlp/submission_{TASK}.csv", index = False)